# CP201A Lab 6: Effective Charts and Tables

**Fall 2026** (October 7 and October 9)

Labs 4 and 5 gave you the numbers for Assignment 1: estimates, margins of error, and
significance tests for your neighborhood and its city or county. This lab turns those numbers
into exhibits a reader can use: charts with error bars, a table with counts and percents, and
the note that goes under each one.

The first half of lab is a short charts module (Sections 1 to 5). The rest is work time on
Assignment 1 with your GSI. Wednesday's recorded lecture walks through Sections 2 and 7, so you
can pause and rewind it at home.

You can make your exhibits two ways, and both are fine for Assignment 1:

* **In Python** (Sections 2 to 6): charts saved as image files you paste into your memo.
* **In Excel** (Section 7): this notebook saves tidy CSV files, and Section 7 walks through the
  chart and error bars in Excel.

Either way, start with Section 1.

## Learning objectives

**Everyone**
* Turn a default chart into a memo-ready exhibit: labels in words, an axis that names the
  universe, data labels, error bars, and a consistent color for your neighborhood
* Write the note that goes under an exhibit: source, geography, MOE method, and test
* Build a table with counts, percents, MOEs, and Z values
* Choose a chart that fits the data: grouped bars, a 100 percent stacked bar, or two periods
* Export clean CSV files for Excel, if that is your route

**If you want more**
* Build a neighborhood mean from an aggregate table, and rebuild a neighborhood median from its
  income distribution (Section 8)

**Bring:** the CSV files Lab 4 saved for your neighborhood, and your Lab 5 notebook with
Exercise #5 done (Section 6.1 below shows how to save its numbers for the Part II charts).

**What you submit this week:** nothing from this notebook. Assignment 1 is one PDF on bCourses,
due Sunday, October 11, at 11:59 p.m.

## 0. Before we begin

Sections 1 to 7 use only the CSV files you already have, so you do not need your Census API key
today unless you try the optional Section 8.

Run the cell below first, every time you open this notebook. It loads the packages and sets the
colors used in every chart: **coral for your neighborhood, dark blue for the city or county,
light blue for a third geography or an earlier period.** Keeping one color per place across
every exhibit is one of the easiest ways to make a memo read as a whole.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

NBHD_COLOR = '#E85C34'    # coral: your neighborhood
COMP_COLOR = '#1B4A82'    # dark blue: the city or county
THIRD_COLOR = '#A9C3E0'   # light blue: a third geography, or the earlier period

plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,      # no box around the plot
    'axes.spines.right': False,
    'axes.titlesize': 12,
})
print('Ready.')

## 1. Your files from Lab 4

### 1.1 Set your parameters

Same idea as Lab 4 Section 1.1 and Lab 5 Section 1.1: change these lines once, and every cell
below uses them.

* `NEIGHBORHOOD_NAME` must match what you used in Lab 4, because the file names are built from it
  (`West Oakland` becomes `lab4_west_oakland_...csv`).
* `LAB4_FOLDER` is where Lab 4 saved its CSVs. If you uploaded them somewhere else, Lab 5
  Section 1.1 explains how to upload and what to change here. If they are in this Week 7
  folder, set `LAB4_FOLDER = ''`.
* `COMP` is your comparison geography, copied exactly as it appears in the files (Section 1.2
  prints the labels).
* `CONFIDENCE` is the one level you chose in Lab 5 Section 3.2 for every test in the memo.

**If your Lab 4 CSVs came from the West Oakland defaults, they are not your data.** Run Lab 4
again with your own tract list first (Lab 4 Section 1.1, then the whole notebook including
Sections 10 and 11).

In [ ]:
NEIGHBORHOOD_NAME = 'West Oakland'
LAB4_FOLDER = '../Week05_Lab_AggregatingMOEs/'
LAB5_FOLDER = '../Week06_Lab_StatisticalSignificance/'
COMP = 'Oakland city, California'     # your comparison geography, exactly as printed in Section 1.2
CONFIDENCE = 90                       # 90, 95, or 99: the level you chose in Lab 5 Section 3.2

CRITICAL = {90: 1.645, 95: 1.960, 99: 2.576}
safe_name = NEIGHBORHOOD_NAME.lower().replace(' ', '_')
NBHD = NEIGHBORHOOD_NAME

### 1.2 Read the files

`read_lab4()` reads one Lab 4 comparison file. The race and ethnicity file is
`lab4_<name>_comparison_wide.csv`; the others have the table's short name in the middle
(`_tenure_`, `_poverty_`, `_commute_`, or whatever you called your own tables in Lab 4
Section 11). A file that is not there comes back as `None`, with a message, rather than an
error.

In [ ]:
def read_lab4(kind=''):
    '''Read one Lab 4 comparison CSV and index it by NAME.

    kind: '' for race and ethnicity, or the middle of the file name ('tenure', 'poverty', ...)
    Returns a DataFrame, or None if the file is not there.
    '''
    middle = f'_{kind}' if kind else ''
    path = f'{LAB4_FOLDER}lab4_{safe_name}{middle}_comparison_wide.csv'
    if not os.path.exists(path):
        print(f'Not found: {path}')
        return None
    print(f'Read: {path}')
    return pd.read_csv(path).set_index('NAME')


race = read_lab4()
tenure = read_lab4('tenure')
poverty = read_lab4('poverty')

print('\nAll of your Lab 4 files in that folder:')
for path in sorted(glob.glob(f'{LAB4_FOLDER}lab4_{safe_name}*.csv')):
    print('   ', os.path.basename(path))

if race is not None:
    print('\nRow labels (copy COMP from this list exactly):')
    for label in race.index:
        print('   ', repr(label))
else:
    print('\nNo race and ethnicity file. Check NEIGHBORHOOD_NAME and LAB4_FOLDER, or upload your',
          'Lab 4 CSVs (Lab 5 Section 1.1 shows how).')

### 1.3 Tools from Lab 5

These are copied from Lab 5 without changes (Sections 2.1 and 4), plus one small new helper.
`short_name()` turns the Census Bureau's labels into the names a reader expects: `Oakland city,
California` becomes `Oakland`, and `Alameda County, California` becomes `Alameda County`.

In [ ]:
# ---- Copied from Lab 5. Nothing new here. ----
def add_standard_errors(df):
    '''Add a <col>_se column (MOE / 1.645) for every <col>_moe column in df.'''
    df = df.copy()
    for col in df.columns:
        if col.endswith('_moe'):
            df[col[:-4] + '_se'] = df[col] / 1.645
    return df


def z_statistic(df, col, place_1, place_2):
    '''Two-sample Z for the difference between two ACS estimates (rows place_1 and place_2).'''
    x1, x2 = df.loc[place_1, col], df.loc[place_2, col]
    se1, se2 = df.loc[place_1, col + '_se'], df.loc[place_2, col + '_se']
    return abs(x1 - x2) / (se1**2 + se2**2)**0.5


def significance(z, level=None):
    '''Pass or fail at your chosen confidence level.'''
    if level is None:
        level = CONFIDENCE
    if z > CRITICAL[level]:
        return f'significant at {level} percent'
    return f'not significant at {level} percent'


# ---- New in Lab 6 ----
def short_name(name):
    '''"Oakland city, California" -> "Oakland"; "Alameda County, California" -> "Alameda County".'''
    return str(name).split(',')[0].replace(' city', '').strip()


print(short_name(COMP))

## 2. From the Lab 5 preview chart to a memo exhibit

### 2.1 Where Lab 5 left off

This is the chart from Lab 5 Section 7, for your Question 2 share (the renter share if you have
a tenure file, otherwise the poverty rate). It shows the right numbers, and it has error bars.
It is still a default chart: the place names are the Census Bureau's, the axis says only
"Percent," both bars are the same color, and the title is a variable name.

In [ ]:
if tenure is not None:
    q2, Q2_SHARE, Q2_TABLE = tenure, 'pct_renter', 'B25003'
else:
    q2, Q2_SHARE, Q2_TABLE = poverty, 'pct_below_poverty', 'B17001'

ax = q2.loc[[NBHD, COMP]].plot.bar(
    y=Q2_SHARE, yerr=f'{Q2_SHARE}_moe', legend=False, capsize=4, color='#1B4A82', figsize=(5, 4)
)
ax.set_ylabel('Percent')
ax.set_xlabel('')
ax.set_title(f'{Q2_SHARE}, 2020 to 2024 (error bars: 90 percent MOE)')
plt.xticks(rotation=0)
plt.tight_layout()

### 2.2 Six changes, one at a time

The cell below draws the same numbers with six changes, numbered in the comments:

1. **Place names in words**, using `short_name()`.
2. **Your neighborhood in its own color**, the same coral in every exhibit.
3. **Error bars** from the MOE column, a little heavier so they print well.
4. **Data labels** on the bars, so no one reads values off gridlines.
5. **An axis label that names the universe.** "Percent" of what? Of occupied housing units for
   tenure (B25003), of people for whom poverty status is determined for poverty (B17001). The
   universe is the `_001E` total in your Lab 4 dictionary.
6. **No title inside the chart.** The figure number and title go in your document, above the
   image, where spellcheck works and you can edit them. The note goes under the image.

Then it saves the chart as a PNG at 300 dots per inch, sharp enough to print. The file appears in
this folder in the file browser on the left.

In [ ]:
UNIVERSE = {
    'pct_renter': 'Percent of occupied housing units',
    'pct_owner': 'Percent of occupied housing units',
    'pct_below_poverty': 'Percent of people for whom poverty status is determined',
}

rows = [NBHD, COMP]
values = q2.loc[rows, Q2_SHARE]
moes = q2.loc[rows, f'{Q2_SHARE}_moe']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    [short_name(r) for r in rows],                     # 1. place names in words
    values,
    color=[NBHD_COLOR, COMP_COLOR],                    # 2. your neighborhood in its own color
    yerr=moes, capsize=6, error_kw={'elinewidth': 1.5},   # 3. error bars from the MOE column
    width=0.6,
)
ax.bar_label(bars, labels=[f'{v:.1f}' for v in values],   # 4. data labels
             label_type='center', color='white', fontsize=12, fontweight='bold')
ax.set_ylabel(UNIVERSE[Q2_SHARE])                      # 5. the axis names the universe
ax.set_ylim(0, (values + moes).max() * 1.15)           # bars start at zero
# 6. no title inside the chart: it goes in your document

fig.tight_layout()
fig.savefig(f'lab6_{safe_name}_q2_share.png', dpi=300, bbox_inches='tight')
print(f'Saved lab6_{safe_name}_q2_share.png')

### 2.3 The same thing as a function

`bar_with_moe()` packages Section 2.2 so you can make the next chart in one line. It works for
any table with an estimate column and a matching `_moe` column, with places or periods as rows.
For dollar values (median rent, median income), pass `dollars=True` and the labels and axis get
dollar signs and commas.

In [ ]:
def bar_with_moe(df, col, rows, ylabel, filename=None, colors=None, dollars=False, figsize=(6, 4)):
    '''Bar chart with 90 percent MOE error bars and data labels.

    df: DataFrame indexed by place (or period) name, with col and col + '_moe'
    rows: the row labels to draw, in order (your neighborhood first)
    ylabel: the universe, in words
    filename: if given, saves a 300 dpi PNG
    colors: one color per bar (default: neighborhood coral, then dark blue, then light blue)
    dollars: True for dollar values
    '''
    values = df.loc[rows, col].astype(float)
    moes = df.loc[rows, f'{col}_moe'].astype(float)
    if colors is None:
        colors = [NBHD_COLOR, COMP_COLOR, THIRD_COLOR][:len(rows)]
    fmt = '${:,.0f}' if dollars else '{:.1f}'

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar([short_name(r) for r in rows], values, color=colors,
                  yerr=moes, capsize=6, error_kw={'elinewidth': 1.5}, width=0.6)
    label_colors = ['black' if c == THIRD_COLOR else 'white' for c in colors]
    for bar, v, lc in zip(bars, values, label_colors):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, fmt.format(v),
                ha='center', va='center', color=lc, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, (values + moes).max() * 1.15)
    if dollars:
        ax.yaxis.set_major_formatter(StrMethodFormatter('${x:,.0f}'))
    fig.tight_layout()
    if filename:
        fig.savefig(filename, dpi=300, bbox_inches='tight')
        print(f'Saved {filename}')
    return fig, ax


# The same chart as Section 2.2, now with the county added as a third bar if it is in your file
places = [NBHD, COMP] + [r for r in q2.index if r not in (NBHD, COMP)][:1]
fig, ax = bar_with_moe(q2, Q2_SHARE, places, UNIVERSE[Q2_SHARE])

### 2.4 The note under the exhibit

Every exhibit gets a note: source table and years, **the universe** (who or what the percent
is of), geography, how the neighborhood value was made, what the error bars show, and the test
with your confidence level. Lab 5 Section 5.2 is the checklist. Write MOEs with a ± sign, and
remember that the MOE on a share is in **percentage points** (Lab 4 Section 4): a renter share of
70.7 percent ± 3.0 means somewhere between 67.7 and 73.7 percent. The cell below writes a draft from your numbers, so the Z in the note always matches
the Z you computed. Copy it into your document under the chart and edit the wording; the
neighborhood definition itself (boundary source and tract list) goes once, in your introduction.

The error bars are 90 percent MOEs, because that is the level the Census Bureau publishes. If
you chose 95 or 99 percent for your tests, say so in the note; the bars still show 90 percent.

In [ ]:
UNIVERSE_WORDS = {
    'pct_renter': 'occupied housing units',
    'pct_owner': 'occupied housing units',
    'pct_below_poverty': 'people for whom poverty status is determined',
}

q2_se = add_standard_errors(q2)
z_q2 = z_statistic(q2_se, Q2_SHARE, NBHD, COMP)
n_tracts = int(q2.loc[NBHD, 'n_tracts'])

note = (
    f'Source: U.S. Census Bureau, American Community Survey 2020 to 2024 5-year estimates, '
    f'Table {Q2_TABLE}. Universe: {UNIVERSE_WORDS[Q2_SHARE]}. '
    f'Note: {NBHD} is {n_tracts} census tracts (2020); see the Introduction. Neighborhood counts '
    f'are summed across tracts, with MOEs combined by root sum of squares; the share MOE uses the '
    f'proportion formula. Error bars show 90 percent margins of error (\u00b1, in percentage points). '
    f'Difference tested with the '
    f'two-estimate Z test (U.S. Census Bureau, 2020, Section 7): Z = {z_q2:.2f}, '
    f'{significance(z_q2)}.'
)
print(note)

### 2.5 Writing about the comparison

The two or three sentences under each exhibit should do more than say which number is higher, or
report only the test. A reader needs four things, in roughly this order:

1. **Both estimates, with their MOEs**, in words: "70.7 percent of West Oakland households rent
   (±3.0 percentage points), compared with 57.7 percent in Oakland (±0.8)."
2. **How big the gap is**: the difference in percentage points, and a comparison a reader can
   picture ("about 1.2 times the city's share," "roughly twice," "about one in five").
3. **The test**: Z and your confidence level, and what it lets you say.
4. **What it might mean for your planning issue**, without claiming a cause the data can't show.

`describe_comparison()` drafts the first three from your numbers (add `periods=True` for a
Part II change, and `dollars=True` for dollar values). Notice that it only offers a "times"
comparison when the difference passes the test. It is a starting point for your
own sentences: rewrite it in your voice, and add the fourth part yourself.

In [ ]:
def describe_comparison(df, col, a, b, dollars=False, periods=False):
    # Draft sentences: both estimates with MOEs, the size of the gap, and the test.
    # periods=True when a and b are the earlier and later period (Part II).
    df = add_standard_errors(df)
    x1, x2 = df.loc[a, col], df.loc[b, col]
    m1, m2 = df.loc[a, f'{col}_moe'], df.loc[b, f'{col}_moe']
    z = z_statistic(df, col, a, b)
    passes = z > CRITICAL[CONFIDENCE]
    if dollars:
        v1, v2 = f'${x1:,.0f} (±${m1:,.0f})', f'${x2:,.0f} (±${m2:,.0f})'
        gap = f'${abs(x2 - x1):,.0f} ({(x2 - x1) / x1 * 100:+.1f} percent)'
    else:
        v1, v2 = f'{x1:.1f} percent (±{m1:.1f} percentage points)', f'{x2:.1f} percent (±{m2:.1f})'
        gap = f'{abs(x2 - x1):.1f} percentage points'
    if periods:
        direction = 'rose' if x2 > x1 else 'fell'
        text = f'The estimate {direction} from {v1} in {a} to {v2} in {b}, a change of {gap}. '
    else:
        text = f'{short_name(a)}: {v1}; {short_name(b)}: {v2}. The difference is {gap}'
        if passes and not dollars and min(x1, x2) > 0:
            text += f"; {short_name(a)}'s share is about {x1 / x2:.1f} times {short_name(b)}'s. "
        else:
            text += '. '
    text += f'Z = {z:.2f}, {significance(z)}.'
    if not passes:
        text += (' Not significant: the data cannot tell this difference apart from zero, '
                 'so do not describe one as higher or lower, or the change as a rise or fall.')
    return text

print(describe_comparison(q2, Q2_SHARE, NBHD, COMP))
print()
print(describe_comparison(race, 'pct_nh_black', NBHD, COMP))
print()
print(describe_comparison(race, 'pct_nh_white', NBHD, COMP))

In [ ]:
# EXERCISE #1: make the memo-ready chart and its note for your own Question 2 share (or for the
# other share, if you did both). Use bar_with_moe() with your neighborhood first and your
# comparison geography second, save it with a filename, and print the note. Then read the note
# against Lab 5 Section 5.2: is anything missing for your exhibit (combined categories, excluded
# tracts)? Add it. Then run describe_comparison() and write your 2 to 3 sentences in your own words.

## 3. Race and ethnicity: all eight groups

Part I Question 1 asks who lives in the neighborhood, using B03002. Keep all eight groups,
including some other race and two or more races. If you decide to combine any, that is a choice
you make on purpose and say in the note (Lab 4 Section 3, Exercise #3).

Eight groups and two places make a crowded vertical bar chart. A **horizontal** grouped bar
handles it better: the labels read left to right, and sorting by your neighborhood's shares puts
the largest group at the top. Data labels sit past the end of each error bar, so the small groups
still read. Each bar's MOE is drawn, and some will be wide relative to the estimate; that is the
honest picture for a small group in a small area.

In [ ]:
GROUPS = ['hispanic', 'nh_white', 'nh_black', 'nh_native', 'nh_asian', 'nh_pi', 'nh_1other', 'nh_multi']
SHORT_LABELS = {
    'hispanic':  'Hispanic or Latino',
    'nh_white':  'White',
    'nh_black':  'Black',
    'nh_native': 'American Indian, Alaska Native',
    'nh_asian':  'Asian',
    'nh_pi':     'Native Hawaiian, Pacific Islander',
    'nh_1other': 'Some other race',
    'nh_multi':  'Two or more races',
}


def grouped_hbar(df, groups, rows, labels, xlabel, filename=None, colors=None, figsize=(8, 6)):
    '''Horizontal grouped bars for several shares, two or three places, with MOE error bars.

    Sorted by the first place's shares (largest at the top). Data labels sit past the error bars.
    '''
    if colors is None:
        colors = [NBHD_COLOR, COMP_COLOR, THIRD_COLOR][:len(rows)]
    order = sorted(groups, key=lambda g: df.loc[rows[0], f'pct_{g}'])
    y = np.arange(len(order))
    h = 0.8 / len(rows)
    fig, ax = plt.subplots(figsize=figsize)
    for i, (r, color) in enumerate(zip(rows, colors)):
        pos = y + (len(rows) - 1 - 2 * i) * h / 2          # first place on top within each group
        vals = np.array([df.loc[r, f'pct_{g}'] for g in order], dtype=float)
        errs = np.array([df.loc[r, f'pct_{g}_moe'] for g in order], dtype=float)
        ax.barh(pos, vals, height=h, color=color, xerr=errs, capsize=3,
                error_kw={'elinewidth': 1}, label=short_name(r))
        for p, v, e in zip(pos, vals, errs):
            ax.text(v + e + 0.6, p, f'{v:.1f}', va='center', fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels([labels[g] for g in order])
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, max(df.loc[r, f'pct_{g}'] + df.loc[r, f'pct_{g}_moe'] for r in rows for g in groups) * 1.15)
    handles, leg_labels = ax.get_legend_handles_labels()
    ax.legend(handles, leg_labels, frameon=False, loc='lower right')
    fig.tight_layout()
    if filename:
        fig.savefig(filename, dpi=300, bbox_inches='tight')
        print(f'Saved {filename}')
    return fig, ax


fig, ax = grouped_hbar(race, GROUPS, [NBHD, COMP], SHORT_LABELS, 'Percent of population',
                       filename=f'lab6_{safe_name}_race.png')

The note for this chart needs one thing the Section 2.4 note does not: **groups other than
Hispanic or Latino are non-Hispanic** (that is how B03002 is built). It also needs the test
results for the groups you write about. With eight tests in one chart, the cleanest place for
every Z is a table, which is Section 4.

In [ ]:
# EXERCISE #2: make the race and ethnicity chart for your neighborhood and comparison geography,
# and write its note. Add your county as a third place if you compared with both. If you combined
# any groups in Lab 4, change GROUPS and SHORT_LABELS to match and say so in the note.

## 4. Tables

A table is often the better exhibit when readers will look up numbers, when every row needs its
MOE and Z, or when some values are too small to see as bars. The guidelines from Monday's lecture
(Schwabish, 2020) come down to a few habits: labels in words, numbers right-aligned with commas
and the same decimals down a column, counts beside percents, and a note under the table.

`share_table()` builds that table for any set of shares from Lab 4: the neighborhood's count,
percent, and MOE; the comparison's percent and MOE; and Z, with an asterisk where the difference
passes at your confidence level. The numbers come out as formatted text, ready to paste.

In [ ]:
def share_table(df, groups, labels, nbhd, comp, total='total', total_label='Total'):
    '''A memo-ready table of shares for two places, with Z and an asterisk at your level.'''
    df = add_standard_errors(df)
    n, c = short_name(nbhd), short_name(comp)
    out = []
    for g in groups:
        z = z_statistic(df, f'pct_{g}', nbhd, comp)
        star = '*' if z > CRITICAL[CONFIDENCE] else ''
        out.append({
            '': labels[g],
            f'{n} count': f'{df.loc[nbhd, g]:,.0f}',
            f'{n} %': f"{df.loc[nbhd, f'pct_{g}']:.1f}",
            f'{n} MOE': f"\u00b1{df.loc[nbhd, f'pct_{g}_moe']:.1f}",
            f'{c} %': f"{df.loc[comp, f'pct_{g}']:.1f}",
            f'{c} MOE': f"\u00b1{df.loc[comp, f'pct_{g}_moe']:.1f}",
            'Z': f'{z:.2f}{star}',
        })
    out.append({'': total_label, f'{n} count': f'{df.loc[nbhd, total]:,.0f}', f'{n} %': '100.0',
                f'{n} MOE': '', f'{c} %': '100.0', f'{c} MOE': '', 'Z': ''})
    return pd.DataFrame(out)


race_table = share_table(race, GROUPS, SHORT_LABELS, NBHD, COMP, total='total', total_label='Total population')
race_table.to_csv(f'lab6_{safe_name}_race_table.csv', index=False)
print(f'Saved lab6_{safe_name}_race_table.csv')
race_table

**Getting the table into your memo.** Download the CSV (right-click it in the file browser on
the left, then Download), open it in Excel or Google Sheets, select the cells, and paste into
Word or Google Docs. Then right-align the number columns, set one font, lighten the lines, and
add the title above ("Table 1. Race and ethnicity, ...") and the note below. In the note: the
source table and years, the universe, how the neighborhood was built, "MOEs at the 90 percent
level, in percentage points," and what the asterisk means ("* Significant at the 90 percent level, two-estimate Z test").

A total row with counts tells the reader how many people the percents describe: 20 percent of 50
people and 20 percent of 50,000 are very different planning facts.

In [ ]:
# EXERCISE #3: build a table for one more Part I question with share_table(). For tenure:
# groups=['owner', 'renter'], labels={'owner': 'Owner occupied', 'renter': 'Renter occupied'},
# total='units', total_label='Occupied housing units'. For poverty: groups=['below_poverty'],
# labels={'below_poverty': 'Below the poverty level'}, total='pov_universe',
# total_label='Population for whom poverty status is determined'. Save it as a CSV.

## 5. Parts of a whole: a 100 percent stacked bar

Owners and renters add up to 100 percent of occupied housing units, so two separate bars repeat
the same information. A 100 percent stacked bar shows the whole in one shape. Use it for any
set of categories that add to the universe: tenure, commute modes, income bands.

Error bars do not work on stacked segments, so put the MOEs in the note or in a table beside
the chart. The segment colors stand for categories (owner, renter), so they are deliberately
different from the place colors used everywhere else. This chart only runs if you have a tenure
file.

In [ ]:
def stacked_shares(df, cols, rows, col_labels, xlabel, filename=None, colors=None, figsize=(8, 3)):
    '''Horizontal 100 percent stacked bars: one bar per place, one segment per share column.'''
    if colors is None:
        colors = ['#2A9D8F', '#F2B134', THIRD_COLOR, '#6B7C93', '#B5D9D3'][:len(cols)]   # categories, not places
    fig, ax = plt.subplots(figsize=figsize)
    names = [short_name(r) for r in rows][::-1]          # first place on top
    left = np.zeros(len(rows))
    for col, lab, color in zip(cols, col_labels, colors):
        vals = np.array([df.loc[r, col] for r in rows][::-1], dtype=float)
        ax.barh(names, vals, left=left, color=color, label=lab, height=0.6)
        for i, v in enumerate(vals):
            if v >= 5:
                ax.text(left[i] + v / 2, i, f'{v:.1f}', ha='center', va='center',
                        color='black' if color in (THIRD_COLOR, '#F2B134', '#B5D9D3') else 'white', fontweight='bold')
        left += vals
    ax.set_xlim(0, 100)
    ax.set_xlabel(xlabel)
    ax.legend(frameon=False, ncol=len(cols), loc='upper center', bbox_to_anchor=(0.5, -0.35))
    fig.tight_layout()
    if filename:
        fig.savefig(filename, dpi=300, bbox_inches='tight')
        print(f'Saved {filename}')
    return fig, ax


if tenure is not None:
    tenure_places = [NBHD, COMP] + [r for r in tenure.index if r not in (NBHD, COMP)][:1]
    fig, ax = stacked_shares(tenure, ['pct_owner', 'pct_renter'], tenure_places,
                             ['Owner occupied', 'Renter occupied'], 'Percent of occupied housing units',
                             filename=f'lab6_{safe_name}_tenure_stacked.png')
else:
    print('No tenure file, so no stacked bar here. The function works the same on any shares that add to 100.')

## 6. Two periods (Part II)

Part II compares 2015 to 2019 with 2020 to 2024. The chart is the same `bar_with_moe()`, with
periods as the rows instead of places, the earlier period in light blue and the later one in dark
blue. Two periods are two snapshots, so use bars, never a line (a line implies the years in
between were measured).

### 6.1 Save your Part II numbers from Lab 5

Your own Part II test is in **Lab 5, Exercise #5** (or **Exercise #6**, if your question is about
dollars). Lab 5 printed the result but did not save it, and this notebook can only chart what is
saved in a file. So this step means **going back to your Lab 5 notebook for a few minutes**:
you add one line to your Exercise #5 cell, run it, and then come back here to Section 6.2. Do
this once for each of your two Part II questions.

1. Open `Lab5_Notebook_Student.ipynb` in your Week 6 folder
   (`PlanningMethods2026/Week06_Lab_StatisticalSignificance/`).
2. Click in your Exercise #5 cell, the code cell that begins `# EXERCISE #5:` (just after
   Section 6.2). For a dollar question, use the cell that begins `# EXERCISE #6:` (in
   Section 6.3) instead.
3. From the menu, choose **Run > Run All Above Selected Cell**, so the setup, your parameters, and
   the functions are loaded again.
4. In that cell, find the line that builds your two-period table: the one that contains
   `.set_index('period')`. (In the Section 6.1 example that table is called `city_change`; in
   6.2 it is `nbhd_change`; yours may have its own name.)
5. Click at the end of that line, press Enter, and type this line, putting your table's name
   where it says `my_change`:

   ```python
   my_change.to_csv(f'lab5_{safe_name}_part2_q1.csv')
   ```

   For your second Part II question, use `part2_q2` in the file name instead of `part2_q1`.
   Keep the new line **above** any last line that only displays the table (like
   `city_change[['NAME', ...]].round(2)`), so the table still prints.
6. Run the cell (Shift+Enter). The file appears in the Week 6 folder, which is where
   `LAB5_FOLDER` points.

### 6.2 Chart one Part II question

Set the four lines at the top to match the file you saved, and run the cell. If the file is not
there yet, the cell uses the Oakland renter share typed in from the Lab 5 Section 6.1 printout, so
you can still see the chart. Typed numbers are rounded, so their Z (about 2.63) differs a little
from the 2.61 Lab 5 printed from unrounded values. That is the reason to save the file: the note
should carry the Z from your actual numbers.

In [ ]:
PART2_FILE = f'{LAB5_FOLDER}lab5_{safe_name}_part2_q1.csv'   # part2_q2 for your second question
PART2_COL = 'pct_renter'          # the column you tested in Lab 5 (a pct_ column, or a dollar column)
PART2_YLABEL = 'Percent of occupied housing units'   # the universe, in words
PART2_DOLLARS = False             # True if PART2_COL is in dollars

if os.path.exists(PART2_FILE):
    part2 = pd.read_csv(PART2_FILE).set_index('period')
    print(f'Read: {PART2_FILE}')
else:
    print(f'Not found: {PART2_FILE}')
    print('Using the Oakland renter share typed from Lab 5 Section 6.1 until you save your own file.')
    part2 = pd.DataFrame({
        'period': ['2015 to 2019', '2020 to 2024'],
        'NAME': ['Oakland city, California'] * 2,
        'pct_renter': [59.26, 57.67],
        'pct_renter_moe': [0.58, 0.81],
    }).set_index('period')
    PART2_COL, PART2_YLABEL, PART2_DOLLARS = 'pct_renter', 'Percent of occupied housing units', False

print('Rows in the file:', list(part2.index))

# The last two rows are the earlier and later period. (A dollar table from Lab 5 Section 6.3 has
# three rows: nominal 2019, 2019 in 2024 dollars, and 2024. The last two are the ones to compare.)
PERIODS = list(part2.index)[-2:]
part2 = add_standard_errors(part2)
z_part2 = z_statistic(part2, PART2_COL, *PERIODS)
print(f'Comparing {PERIODS[0]!r} with {PERIODS[1]!r}:')
print(describe_comparison(part2, PART2_COL, *PERIODS, dollars=PART2_DOLLARS, periods=True))

part2_plot = part2.loc[PERIODS].copy()
part2_plot.index = [p.split(' (')[0] for p in PERIODS]      # "2015 to 2019 (in 2024 dollars)" -> "2015 to 2019"
fig, ax = bar_with_moe(part2_plot, PART2_COL, list(part2_plot.index), PART2_YLABEL,
                       colors=[THIRD_COLOR, COMP_COLOR], dollars=PART2_DOLLARS,
                       filename=f'lab6_{safe_name}_part2_{PART2_COL}.png')

### 6.3 Dollars across periods

A dollar comparison needs one more line in its note: whether the 2015 to 2019 values were
adjusted to 2024 dollars (Lab 5 Section 6.3, the R-CPI-U-RS factor 1.2307) or are nominal. The
example below uses Oakland's median gross rent from the Lab 5 Section 6.3 printout, in constant
2024 dollars, typed in. Because it is a published city median, it can be tested. A class-rule
neighborhood median cannot (Lab 5 Section 8). For your own dollar question, use Section 6.2 with
your `part2` file and `PART2_DOLLARS = True`.

In [ ]:
rent = pd.DataFrame({
    'period': ['2015 to 2019 (in 2024 dollars)', '2020 to 2024 (2024 dollars)'],
    'med_rent': [1778, 1979],
    'med_rent_moe': [22, 29],
}).set_index('period')

rent = add_standard_errors(rent)
z_rent = z_statistic(rent, 'med_rent', *rent.index)
print(describe_comparison(rent, 'med_rent', *rent.index, dollars=True, periods=True))

rent.index = ['2015 to 2019', '2020 to 2024']
fig, ax = bar_with_moe(rent, 'med_rent', list(rent.index), 'Median gross rent (2024 dollars)',
                       colors=[THIRD_COLOR, COMP_COLOR], dollars=True)

print('\nDraft note:')
print('Source: U.S. Census Bureau, ACS 5-year estimates for 2015 to 2019 and 2020 to 2024, Table B25064. '
      'Universe: renter-occupied housing units paying cash rent. '
      'Note: 2015 to 2019 values adjusted to 2024 dollars with the R-CPI-U-RS (factor 1.2307; '
      'U.S. Census Bureau, Comparing 2024 ACS Data). Error bars show 90 percent margins of error. '
      f'Change tested with the two-estimate Z test (U.S. Census Bureau, 2020, Section 7): '
      f'Z = {z_rent:.2f}, {significance(z_rent)}.')

In [ ]:
# EXERCISE #4: save both of your Part II questions from Lab 5 (Section 6.1 steps), chart each one
# with the Section 6.2 cell (change PART2_FILE to part2_q2 for the second), and write each note. On the neighborhood path, the note also says how you handled the tract
# lists (the Lab 5 Section 6.0 verdicts: unchanged, split, renumbered). For dollars, say whether
# values are adjusted or nominal. If you compared your neighborhood's change with the city's,
# a grouped chart with place on the axis and period as the color shows that comparison.

## 7. The Excel route

If you would rather design your charts in Excel, this section saves small, tidy CSV files (one
row per place or period, with the estimate and its MOE in plain columns) and walks through the
chart. Excel charts are fine for Assignment 1; the rubric grades the content and labeling.

### 7.1 Save chart-ready CSVs

In [ ]:
def export_for_excel(df, col, rows, filename, label='Place', decimals=1):
    '''Save a tidy two-column table (estimate and MOE) for one share, ready for an Excel chart.'''
    out = pd.DataFrame({
        label: [short_name(r) for r in rows],
        'Estimate': df.loc[rows, col].astype(float).round(decimals).values,
        'MOE': df.loc[rows, f'{col}_moe'].astype(float).round(decimals).values,
    })
    out.to_csv(filename, index=False)
    print(f'Saved {filename}')
    return out


export_for_excel(q2, Q2_SHARE, [NBHD, COMP], f'lab6_{safe_name}_q2_for_excel.csv')

# Race and ethnicity: one row per group, estimate and MOE for each place
race_excel = pd.DataFrame({
    'Group': [SHORT_LABELS[g] for g in GROUPS],
    f'{short_name(NBHD)} %': [round(race.loc[NBHD, f'pct_{g}'], 1) for g in GROUPS],
    f'{short_name(NBHD)} MOE': [round(race.loc[NBHD, f'pct_{g}_moe'], 1) for g in GROUPS],
    f'{short_name(COMP)} %': [round(race.loc[COMP, f'pct_{g}'], 1) for g in GROUPS],
    f'{short_name(COMP)} MOE': [round(race.loc[COMP, f'pct_{g}_moe'], 1) for g in GROUPS],
})
race_excel.to_csv(f'lab6_{safe_name}_race_for_excel.csv', index=False)
print(f'Saved lab6_{safe_name}_race_for_excel.csv')
race_excel

### 7.2 Download the files

In the file browser on the left, right-click each `lab6_..._for_excel.csv` file and choose
**Download**. Datahub is not a backup, so download your PNG files and this notebook too
(File > Download).

### 7.3 A bar chart with error bars in Excel

Menu names differ a little between Windows and Mac and between Excel versions; these follow
current Microsoft 365. If a menu is not where the steps say, the chart tools appear on the
ribbon when the chart is selected (**Chart Design** and **Format**).

1. **Open the CSV** and copy the rows you need to a new sheet (Paste Special > Values). Keep
   the label column, the estimate, and the MOE.
2. **Insert the chart.** Select the label and Estimate columns only (not the MOE). Insert >
   Insert Column or Bar Chart > Clustered Column. For many categories, choose Clustered Bar
   instead.
3. **Add error bars.** With the chart selected: Chart Design > Add Chart Element > Error Bars >
   More Error Bar Options. Under Direction, choose **Both**.
4. **Use your MOEs.** Under Error Amount, choose **Custom**, then **Specify Value**. For the
   Positive Error Value, select the MOE cells with your mouse; do the same for the Negative Error
   Value (Excel wants positive numbers in both boxes).
5. **Fix the axis.** Double-click the vertical axis numbers. Under Axis Options > Bounds, set
   the Minimum to 0. Excel sometimes starts the axis above zero, which exaggerates differences.
6. **Label it.** Chart Design > Add Chart Element > Axis Titles > Primary Vertical, and type the
   universe ("Percent of occupied housing units"). Add Data Labels from the same menu.
7. **Color your neighborhood.** Click once on the bars, then click again on your neighborhood's
   bar alone, and set its fill (Format Data Point > Fill) to the same color you use everywhere.
8. **Delete the default chart title** and any gridlines you do not need. Write the figure number,
   title, and note in your document, not in Excel: Excel does not spellcheck chart text.
9. **Copy the chart into Word or Google Docs.**

**If Excel asks for MOEs one series at a time,** your places are being read as separate
series. Open Chart Design > Select Data and move the places to the horizontal (category) axis,
so one series holds all the estimates; then one MOE range covers every bar.

**For two places across many groups** (the race and ethnicity file), select the Group column
and both % columns, insert a Clustered Bar, and add custom error bars to each series with its
own MOE column.

### 7.4 Tables in Excel

Open `lab6_<name>_race_table.csv` from Section 4. Select the number columns and right-align
them (Home > Align Right); check that each column shows the same number of decimals (Home >
Increase or Decrease Decimal) and that counts have commas (Home > Comma Style, then remove the
decimals). Copy the cells into Word, and add the title above and the note below.

## 8. If you want more: a neighborhood mean, and a rebuilt median

Optional, for the advanced path. Read Lab 4 Section 6 first.

Lab 4 Section 2.2 noted that medians and means do not add across tracts, and Section 6 gave the
class rule for medians: report the published city or county median, the range of your tract
medians, and (if you need one number) the average of the tract medians labeled as an
approximation. This section shows two better ways to get one neighborhood number, using tables
that **are** made of counts and therefore do aggregate:

* **8.2 A mean** from an aggregate table: aggregate household income (B19025) divided by the
  number of households.
* **8.3 A median** rebuilt from the income distribution (B19001) by finding where the middle
  household falls and interpolating within that income band.

These cells pull from the Census API, so they need your key, and they need your tract list
from Lab 4 Section 1.1.

### 8.1 Setup

In [ ]:
%pip install -q census

In [ ]:
# Your tracts, from Lab 4 Section 1.1 (the West Oakland defaults are below)
TRACT_LIST = ['401400', '401500', '401600', '401700', '401800',
              '402200', '402400', '402500', '402600', '402700',
              '410500', '981900', '982000']
STATE = '06'          # California
COUNTY = '001'        # Alameda County
PLACE = '53000'       # Oakland

In [ ]:
from census import Census

try:
    with open(os.path.expanduser('~/census_key.txt')) as f:
        api_key = f.read().strip()
    print('Key loaded. It starts with:', api_key[:4] + '...')
except FileNotFoundError:
    print('No key file found. Open Lab 3, run the cell in Section 0.1 once to save your key, then run this cell again.')

    api_key = None

c = Census(key=api_key) if api_key else None

In [ ]:
# ---- Copied from Lab 4. Nothing new here. ----
def clean_acs(df, id_cols=('NAME', 'GEO_ID', 'state', 'county', 'tract', 'place')):
    '''Convert estimate and MOE columns to numbers and replace jam values.'''
    df = df.copy()
    numeric_cols = [col for col in df.columns if col not in id_cols]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])
    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]
    df[moe_cols] = df[moe_cols].replace(-555555555, 0)
    df[numeric_cols] = df[numeric_cols].replace([-666666666, -222222222, -333333333], np.nan)
    return df


def aggregate_tracts(df, name):
    '''Add up a DataFrame of tracts into one neighborhood row (sum estimates, RSS the MOEs).'''
    numeric_cols = df.select_dtypes('number').columns
    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]
    est_cols = [col for col in numeric_cols if col not in moe_cols]
    estimates = df[est_cols].sum()
    moes = (df[moe_cols]**2).sum()**0.5
    row = pd.concat([estimates, moes])
    out = pd.DataFrame(row).transpose()
    out.insert(0, 'NAME', name)
    out.insert(1, 'n_tracts', len(df))
    return out


def pull_geo(geo_for, geo_in, variables, year=2024):
    '''Pull one geography for one year and clean it.'''
    df = pd.DataFrame(
        c.acs5.get(list(variables.keys()), {'for': geo_for, 'in': geo_in}, year=year)
    ).rename(columns=variables)
    return clean_acs(df)

print('Toolkit loaded.')

### 8.2 A neighborhood mean from an aggregate

B19025 is the total income of all households in a geography, added up. Aggregates are counts
(of dollars), so they sum across tracts with the same root-sum-of-squares MOE as any count.
Dividing by the number of households gives mean household income for the neighborhood.

The MOE uses the **ratio** formula (U.S. Census Bureau, 2020, Section 8), with addition under the root,
because the numerator (dollars) is not a subset of the denominator (households):

$$MOE_{mean} = \frac{\sqrt{MOE_{agg}^2 + \bar{x}^2 \cdot MOE_{hh}^2}}{households}$$

A mean and a median answer different questions. A few high incomes pull the mean up, so in most
places the mean is well above the median. Whichever you report, call it by its name.

In [ ]:
mean_vars = {
    'NAME': 'NAME', 'GEO_ID': 'GEO_ID',
    'B19001_001E': 'households', 'B19001_001M': 'households_moe',
    'B19025_001E': 'agg_income', 'B19025_001M': 'agg_income_moe',
}

def add_mean(df, agg, count, name):
    '''Add a mean (agg / count) and its ratio MOE to df.'''
    df = df.copy()
    xbar = df[agg] / df[count]
    df[name] = xbar
    df[f'{name}_moe'] = (df[f'{agg}_moe']**2 + xbar**2 * df[f'{count}_moe']**2)**0.5 / df[count]
    return df

mean_all = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', mean_vars)
mean_tracts = mean_all[mean_all['tract'].isin(TRACT_LIST)].copy()
print(f'{len(mean_tracts)} of {len(TRACT_LIST)} tracts found.')
missing = mean_tracts[mean_tracts['agg_income'].isna()]['tract'].tolist()
if missing:
    print('No aggregate income published for:', missing, '(left out of both the income total and the household count; say so in the note)')

mean_nbhd = add_mean(aggregate_tracts(mean_tracts.dropna(subset=['agg_income']), NEIGHBORHOOD_NAME),
                     'agg_income', 'households', 'mean_income')
mean_city = add_mean(pull_geo(f'place:{PLACE}', f'state:{STATE}', mean_vars), 'agg_income', 'households', 'mean_income')
means = pd.concat([mean_nbhd, mean_city], ignore_index=True).set_index('NAME')
means = add_standard_errors(means)

z_mean = z_statistic(means, 'mean_income', NEIGHBORHOOD_NAME, means.index[1])
print(f'Mean household income: Z = {z_mean:.2f}, {significance(z_mean)}')
means[['households', 'agg_income', 'mean_income', 'mean_income_moe']].round(0)

The neighborhood row above drops any tract without a published aggregate from **both** the
numerator and the denominator, so the mean describes the tracts that have one. Say in the note
which tracts were left out.

The Census Bureau describes MOEs built this way as approximations, so say that in the note too
("MOE for the derived mean calculated with the ratio formula; approximate").

### 8.3 Rebuilding a median from the distribution

B19001 counts households in 16 income bands. Those counts aggregate, so the neighborhood's
distribution is just the tract distributions added up. The median household is the one halfway
through that distribution: find the band where the running total passes half of all households,
then estimate how far into the band it falls, assuming households are spread evenly across the
band (linear interpolation).

This gives a better neighborhood median than averaging tract medians, and it is still an
approximation. The Census Bureau computes its published medians from finer income categories
than B19001 publishes, so a rebuilt median will not match a published one exactly. The cell
checks that by rebuilding the city's median the same way and printing it beside the published
city value. The Bureau's method for an MOE on a rebuilt median is beyond this lab, so report the
rebuilt median as an approximation, without an MOE, and do not test it.

In [ ]:
# B19001 lines 002 to 017: lower and upper bounds of each income band (the top band has no upper bound)
BANDS = [(0, 10000), (10000, 15000), (15000, 20000), (20000, 25000), (25000, 30000),
         (30000, 35000), (35000, 40000), (40000, 45000), (45000, 50000), (50000, 60000),
         (60000, 75000), (75000, 100000), (100000, 125000), (125000, 150000), (150000, 200000),
         (200000, None)]
dist_vars = {'NAME': 'NAME', 'GEO_ID': 'GEO_ID', 'B19001_001E': 'households'}
for i in range(len(BANDS)):
    dist_vars[f'B19001_{i + 2:03d}E'] = f'band_{i + 2:02d}'
BAND_COLS = [f'band_{i + 2:02d}' for i in range(len(BANDS))]


def median_from_bands(counts, bands=BANDS):
    '''Median by linear interpolation within the band where the running total passes half.'''
    counts = np.array(counts, dtype=float)
    half = counts.sum() / 2
    running = 0.0
    for n, (low, high) in zip(counts, bands):
        if running + n >= half:
            if high is None:
                return None          # the median is in the top band: report "$200,000 or more"
            return low + (half - running) / n * (high - low)
        running += n
    return None


dist_all = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', dist_vars)
dist_tracts = dist_all[dist_all['tract'].isin(TRACT_LIST)]
dist_city = pull_geo(f'place:{PLACE}', f'state:{STATE}', dist_vars)
city_published = pull_geo(f'place:{PLACE}', f'state:{STATE}',
                          {'NAME': 'NAME', 'B19013_001E': 'med_income', 'B19013_001M': 'med_income_moe'})

nbhd_counts = dist_tracts[BAND_COLS].sum()
nbhd_median = median_from_bands(nbhd_counts)
city_rebuilt = median_from_bands(dist_city[BAND_COLS].iloc[0])

print(f'{NEIGHBORHOOD_NAME}, rebuilt from B19001 ({len(dist_tracts)} tracts): '
      + (f'${nbhd_median:,.0f}' if nbhd_median else '$200,000 or more'))
print(f'{dist_city["NAME"].iloc[0]}, rebuilt the same way: ${city_rebuilt:,.0f}')
print(f'{dist_city["NAME"].iloc[0]}, published (B19013): ${city_published["med_income"].iloc[0]:,.0f} '
      f'(MOE ${city_published["med_income_moe"].iloc[0]:,.0f})')

Compare the city's rebuilt and published medians: the gap tells you roughly how far off the
neighborhood's rebuilt value might be. In your note: "Neighborhood median rebuilt from Table
B19001 by linear interpolation within income bands; approximate, and not tested."

The same method works for rent with B25063 (gross rent), which has its own bands, including a
"no cash rent" line that is left out of the median. Write a new `BANDS` list from the 2024 Groups
page for B25063 if you try it.

## 9. Before you leave

* Your Part I exhibits are saved: race and ethnicity, economic conditions, and housing, each with
  a draft note.
* Your Part II numbers are saved from Lab 5 Exercises #5 and #6 (Section 6.1 here), and at least one
  Part II chart is drawn.
* Your PNG and CSV files and this notebook are **downloaded**, not just saved in Datahub.
* Before Sunday, run your memo against the checklist from Monday's lecture: medians called
  medians, denominators that add to 100 percent of the right universe, every claimed difference
  tested with Z and your one confidence level, numbers in the text that match the exhibits,
  figures numbered in order, and a note under every exhibit.

The rest of lab is work time on Assignment 1. Your GSI is here for whichever step you are stuck
on, including the ones in Labs 4 and 5.

**Assignment 1 is due Sunday, October 11, at 11:59 p.m.**, as one PDF on bCourses.